# 实验报告03：正则表达式与⽹络结构化数据

In [1]:
import hashlib
import os
import platform
import uuid
import sys
# 修改为自己的学号
STUDENT_ID = 20244080302
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
 f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] HRRR
[学号] 20244080302
[机器指纹] c203dca07a7d4af6
[RUN_ID] 20244080302-c203dca07a7d4af6
[Python] 3.13.16


实验1：正则表达式的基础匹配

在本实验开始时导⼊ re。操作任务：
1. 在"cat on mat; later: cats and dogs"上使⽤正则表达式，分别匹配⽂
本中的数字 \d+、\s+和\bcat\b。
2. 在 "cat dog bird CAT" 上使⽤ cat|dog。⽤ re.findall(r"a|aa|aaa",
"aaaa") 和 re.findall(r"aaa|aa|a", "aaaa") 展⽰差异。
3. 在 "ct cat caat caaat" 上分别使⽤ ca*t、ca+t、ca?t、ca{2}t 和
ca{1,2}t，输出匹配列表并说明各量词的含义。

In [1]:
import re

# ---------------- 1. 基础匹配：\d+、\s+、\bcat\b ----------------
text = "cat on mat; later: cats and dogs"

print("文本:", text)
print(r"\d+    ->", re.findall(r"\d+", text))        # 文本中没有数字，返回空列表
print(r"\s+    ->", re.findall(r"\s+", text))        # 空白字符（空格、制表符、换行等）
print(r"\bcat\b ->", re.findall(r"\bcat\b", text))  # 单词边界：只匹配独立的 cat，不匹配 cats

# 换一段带数字的文本，才能看到 \d+ 的效果
print(r"\d+ 在 'later: 3 cats, 12 dogs' 上 ->", re.findall(r"\d+", "later: 3 cats, 12 dogs"))

# ---------------- 2. 分支匹配：cat|dog ----------------
print()
print(r"cat|dog 在 'cat dog bird CAT' 上 ->", re.findall(r"cat|dog", "cat dog bird CAT"))
print(r"  加 re.IGNORECASE ->", re.findall(r"cat|dog", "cat dog bird CAT", flags=re.IGNORECASE))
# re 默认区分大小写，所以 CAT 不会被匹配到

# 分支顺序会影响结果：从左到右扫描，在每个位置先尝试写在前面的分支
print(r"re.findall(r'a|aa|aaa', 'aaaa')   ->", re.findall(r"a|aa|aaa", "aaaa"))
print(r"re.findall(r'aaa|aa|a', 'aaaa')   ->", re.findall(r"aaa|aa|a", "aaaa"))

# ---------------- 3. 量词：*、+、?、{m}、{m,n} ----------------
print()
text3 = "ct cat caat caaat"
patterns = {
    r"ca*t": "a 出现 0 次或多次：ct、cat、caat、caaat 都匹配",
    r"ca+t": "a 出现 1 次或多次：把 ct 排除，caat、caaat 匹配",
    r"ca?t": "a 出现 0 次或 1 次：只匹配 ct 和 cat",
    r"ca{2}t": "a 恰好出现 2 次：只匹配 caat（caaat 中的 a 太多）",
    r"ca{1,2}t": "a 出现 1 到 2 次：匹配 cat 和 caat",
}
print("文本:", text3)
for pattern, meaning in patterns.items():
    print(f"{pattern:<10} -> {str(re.findall(pattern, text3)):<32} {meaning}")


文本: cat on mat; later: cats and dogs
\d+    -> []
\s+    -> [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b -> ['cat']
\d+ 在 'later: 3 cats, 12 dogs' 上 -> ['3', '12']

cat|dog 在 'cat dog bird CAT' 上 -> ['cat', 'dog']
  加 re.IGNORECASE -> ['cat', 'dog', 'CAT']
re.findall(r'a|aa|aaa', 'aaaa')   -> ['a', 'a', 'a', 'a']
re.findall(r'aaa|aa|a', 'aaaa')   -> ['aaa', 'a']

文本: ct cat caat caaat
ca*t       -> ['ct', 'cat', 'caat', 'caaat']   a 出现 0 次或多次：ct、cat、caat、caaat 都匹配
ca+t       -> ['cat', 'caat', 'caaat']         a 出现 1 次或多次：把 ct 排除，caat、caaat 匹配
ca?t       -> ['ct', 'cat']                    a 出现 0 次或 1 次：只匹配 ct 和 cat
ca{2}t     -> ['caat']                         a 恰好出现 2 次：只匹配 caat（caaat 中的 a 太多）
ca{1,2}t   -> ['cat', 'caat']                  a 出现 1 到 2 次：匹配 cat 和 caat


实验2：分组、回溯引⽤与替换

操作任务：
1. 在 "联系：alice@example.com；备选：bob@test.org" 中，⽤捕获组提取每
个邮箱地址的⽤⼾名和域名。
2. 在 "cat cat, dog cat, bird bird" 中，⽤后向引⽤找出连续重复的单词。
3. 将 "2024-05-01\n2024/05/02\n2024.05.03" 中的⽇期从“年-⽉-⽇”形式转
换为“⽉-⽇-年”形式，保留原来的⾏数。可⽤ re.sub()、捕获组和替换表达式完
成；展⽰转换前后⽂本。
4. 将 "联系我：123-456-7890 或 987.654.3210" 中的电话号码统⼀为 (123)
456-7890 形式。

In [3]:
import re

# ---------------- 1. 捕获组：提取邮箱的用户名和域名 ----------------
text1 = "联系：alice@example.com；备选：bob@test.org"
print("原文:", text1)

pattern1 = r"([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)"
print("findall 结果（用户名, 域名）:", re.findall(pattern1, text1))
for match in re.finditer(pattern1, text1):
    print(f"邮箱 = {match.group(0):<18} 用户名 = {match.group(1):<6} 域名 = {match.group(2)}")

# ---------------- 2. 后向引用：找出连续重复的单词 ----------------
text2 = "cat cat, dog cat, bird bird"
print()
print("原文:", text2)

pattern2 = r"\b(\w+)\s+\1\b"   # \1 引用第一个捕获组匹配到的内容
print("重复的单词:", re.findall(pattern2, text2))
for match in re.finditer(pattern2, text2):
    print(f"匹配 {match.group(0)!r} -> 单词 {match.group(1)!r} 连续出现了两次")

# ---------------- 3. 日期格式转换：年-月-日 -> 月-日-年 ----------------
text3 = "2024-05-01\n2024/05/02\n2024.05.03"
print()
print("转换前:")
print(text3)

pattern3 = r"(\d{4})[-/.](\d{2})[-/.](\d{2})"
converted = re.sub(pattern3, r"\2-\3-\1", text3)   # \1=年 \2=月 \3=日
print("转换后:")
print(converted)
print("行数（前后一致）:", len(text3.splitlines()), "->", len(converted.splitlines()))

# ---------------- 4. 电话号码统一格式 ----------------
text4 = "联系我：123-456-7890 或 987.654.3210"
print()
print("原文:", text4)

pattern4 = r"(\d{3})[-.](\d{3})[-.](\d{4})"
print("统一后:", re.sub(pattern4, r"(\1) \2-\3", text4))


原文: 联系：alice@example.com；备选：bob@test.org
findall 结果（用户名, 域名）: [('alice', 'example.com'), ('bob', 'test.org')]
邮箱 = alice@example.com  用户名 = alice  域名 = example.com
邮箱 = bob@test.org       用户名 = bob    域名 = test.org

原文: cat cat, dog cat, bird bird
重复的单词: ['cat', 'bird']
匹配 'cat cat' -> 单词 'cat' 连续出现了两次
匹配 'bird bird' -> 单词 'bird' 连续出现了两次

转换前:
2024-05-01
2024/05/02
2024.05.03
转换后:
05-01-2024
05-02-2024
05-03-2024
行数（前后一致）: 3 -> 3

原文: 联系我：123-456-7890 或 987.654.3210
统一后: 联系我：(123) 456-7890 或 (987) 654-3210


实验3：使⽤ BeautifulSoup 解析 HTML

操作任务：
1. ⽤ BeautifulSoup(html_doc, "html.parser") 构建解析树，输出⻚⾯标题
的⽂本、h1 的标签名与⽂本。
2. 使⽤ find_all() 找出所有 class="course" 的段落和所有链接；逐项输出链
接⽂字与 href 属性。
3. ⽤ get_text() 提取⻚⾯可⻅⽂本；⽐较整个⽂档、单个段落和单个链接的⽂本
结果。

In [4]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

# ---------------- 1. 构建解析树，读取标题与 h1 ----------------
print("页面标题:", soup.title.get_text())
print("h1 标签名:", soup.h1.name, "| h1 文本:", soup.h1.get_text())

# ---------------- 2. find_all() 查找段落和链接 ----------------
courses = soup.find_all("p", class_="course")
links = soup.find_all("a")

print()
print(f"class=\"course\" 的段落共 {len(courses)} 个:")
for p in courses:
    print("  ", p.get_text())

print(f"链接共 {len(links)} 个:")
for a in links:
    print(f"   文字 = {a.get_text():<6} href = {a['href']}")

# ---------------- 3. get_text() 提取可见文本 ----------------
print()
print("整个文档的 get_text():", repr(soup.get_text()))
print("去掉多余空白的版本   :", repr(soup.get_text(strip=True)))
print("单个段落（第一个 p） :", repr(courses[0].get_text()))
print("单个链接（第一个 a） :", repr(links[0].get_text()))


页面标题: 实验课程
h1 标签名: h1 | h1 文本: 数据分析

class="course" 的段落共 2 个:
   第六讲：正则表达式
   第七讲：结构化数据
链接共 2 个:
   文字 = 讲义六    href = https://example.org/lecture06
   文字 = 讲义七    href = https://example.org/lecture07

整个文档的 get_text(): '\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'
去掉多余空白的版本   : '实验课程数据分析第六讲：正则表达式第七讲：结构化数据讲义六讲义七'
单个段落（第一个 p） : '第六讲：正则表达式'
单个链接（第一个 a） : '讲义六'
